# 행사일과 비행사 참고일 비교

이 노트북은 공개용 합성 OD 자료로 전처리와 시간대별 비교 흐름을 확인합니다. 실제 여의도 교통량이나 행사 효과를 나타내지 않습니다.

초기 연구의 행사일은 2023년 10월 7일 토요일이었고 참고일 6개는 모두 일요일이었습니다. 따라서 실제 자료를 다시 분석할 때는 같은 요일의 비행사 날짜를 확보해야 합니다.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from event_traffic.preprocessing import prepare_od, daily_od_profile, compare_event_to_baseline

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sample_path = root / 'data' / 'sample' / 'od_sample.csv'

## 1. 열과 값 검증

`prepare_od`는 필수 열, 음수 이동량, 0 이하 이동시간을 검사합니다. 수단 결측값을 차량으로 바꾸지 않습니다.

In [ ]:
raw = pd.read_csv(sample_path, dtype={'date': str})
od = prepare_od(raw)
od.head()

## 2. 날짜별 프로필

행사일과 참고일 모두 같은 위치·방향·수단 필터를 적용합니다. 이동시간은 OD 이동량으로 가중해 계산합니다.

In [ ]:
profile = daily_od_profile(
    od,
    hdong_code='1156054000',
    direction='arrival',
    mode='car',
)
profile

## 3. 행사일과 참고일 평균 비교

참고일 개수는 고정값으로 나누지 않고 실제 날짜 수로 계산합니다.

In [ ]:
comparison = compare_event_to_baseline(profile, event_date='20231007')
comparison[['time', 'baseline_days', 'event_trip_count', 'baseline_trip_count', 'excess_trip_count', 'delay_min']]

In [ ]:
ax = comparison.plot(
    x='time',
    y=['event_trip_count', 'baseline_trip_count'],
    marker='o',
    figsize=(9, 5),
)
ax.set(title='Synthetic event and reference profiles', xlabel='Arrival time', ylabel='Trip count')
ax.grid(alpha=0.25)
plt.show()

## 해석 범위

실제 자료에서 시간대별 평균 이동시간은 출발지 구성이 달라져도 변할 수 있습니다. 지체를 추정하려면 동일한 `origin × destination × mode` 안에서 행사일과 참고일의 이동시간을 비교한 뒤 집계해야 합니다.